# CROHME Colab One-Click Demo

This notebook runs the full MVP demo in one Colab session:

- base-model inference and evaluation on CROHME benchmark splits
- LoRA fine-tuning on the CROHME train split
- post-training evaluation on the same benchmark splits
- result visualization and side-by-side comparison

The current code path supports `Qwen2-VL`, `Qwen2.5-VL`, and `Qwen3-VL` model IDs in the training and inference backends.

Method summary:

1. Clone the repo from `main`.
2. Install the Python dependencies.
3. Build Colab-safe configs for both pipelines.
4. Run `scripts/run_inference_pipeline.sh`.
5. Run `scripts/run_lora_pipeline.sh`.
6. Generate report figures and compare the aggregated metrics.

Use a GPU runtime before running the notebook.

In [ ]:
REPO_URL = "https://github.com/Zeric-li/HME.git"
WORKDIR = "/content/HME"

!rm -rf "$WORKDIR"
!git clone --depth 1 --branch main "$REPO_URL" "$WORKDIR"
%cd "$WORKDIR"

!python -m pip install -U pip
!pip install -r requirements.txt
!pip install -U "git+https://github.com/huggingface/transformers"

In [ ]:
from pathlib import Path

OUTPUT_ROOT = "/content/outputs"
RUN_BASELINE = True
RUN_LORA = True

INFERENCE_CONFIG = {
    'pipeline_type': 'inference_only',
    'model_id': 'Qwen/Qwen3-VL-4B-Instruct',
    'output_dir': str(Path(OUTPUT_ROOT) / 'qwen3vl-crohme-base'),
    'run_name': 'qwen3vl-crohme-base',
    'eval_dataset_id': 'Neeze/CROHME-full',
    'eval_splits': ['2014', '2016', '2019'],
    'max_eval_samples': 32,
    'eval_batch_size': 1,
    'seed': 42,
    'system_prompt': 'You are a precise handwritten mathematical expression recognition (HMER) system. Extract and Print the expression of the formula in the image using LaTeX format. ',
    'user_prompt': 'Do not wrap the answer in markdown or LaTeX fences. Do not add any text, spaces before or after.',
    'min_pixels': 200704,
    'max_pixels': 602112,
    'max_new_tokens': 128,
    'torch_dtype': 'bfloat16',
    'attn_implementation': 'flash_attention_2',
}

LORA_CONFIG = {
    'pipeline_type': 'lora_train_eval',
    'model_id': 'Qwen/Qwen3-VL-4B-Instruct',
    'output_dir': str(Path(OUTPUT_ROOT) / 'qwen3vl-crohme-lora'),
    'run_name': 'qwen3vl-crohme-lora',
    'train_dataset_id': 'Neeze/CROHME-full',
    'train_split': 'train',
    'eval_dataset_id': 'Neeze/CROHME-full',
    'train_eval_split': '2014',
    'eval_splits': ['2014', '2016', '2019'],
    'max_train_samples': 128,
    'max_eval_samples': 32,
    'eval_batch_size': 1,
    'shuffle_train': True,
    'seed': 42,
    'system_prompt': 'You are a precise handwritten mathematical expression recognition (HMER) system. Extract and Print the expression of the formula in the image using LaTeX format. ',
    'user_prompt': 'Do not wrap the answer in markdown or LaTeX fences. Do not add any text, spaces before or after.',
    'min_pixels': 200704,
    'max_pixels': 602112,
    'max_seq_length': 1024,
    'max_new_tokens': 128,
    'torch_dtype': 'bfloat16',
    'attn_implementation': 'flash_attention_2',
    'gradient_checkpointing': True,
    'per_device_train_batch_size': 2,
    'per_device_eval_batch_size': 1,
    'gradient_accumulation_steps': 8,
    'num_train_epochs': 2,
    'learning_rate': 2.0e-4,
    'weight_decay': 0.0,
    'warmup_ratio': 0.03,
    'lr_scheduler_type': 'cosine',
    'logging_steps': 10,
    'save_steps': 200,
    'eval_steps': 200,
    'save_total_limit': 2,
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'lora_bias': 'none',
    'lora_target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    'report_to': 'none',
}

print('Loaded Colab demo configs:')
print('INFERENCE_CONFIG:')
print(INFERENCE_CONFIG)
print('\nLORA_CONFIG:')
print(LORA_CONFIG)

In [ ]:
from copy import deepcopy
from pathlib import Path
import importlib.util
import json
import torch
import yaml


def resolve_colab_runtime_overrides() -> dict:
    overrides = {
        'torch_dtype': 'float16',
        'attn_implementation': 'sdpa',
    }
    if not torch.cuda.is_available():
        return overrides

    major, _ = torch.cuda.get_device_capability()
    has_flash_attn = importlib.util.find_spec('flash_attn') is not None
    if major >= 8:
        overrides['torch_dtype'] = 'bfloat16'
    if has_flash_attn:
        overrides['attn_implementation'] = 'flash_attention_2'
    return overrides


def build_colab_config(task: str) -> Path:
    config = deepcopy(INFERENCE_CONFIG if task == 'inference_only' else LORA_CONFIG)

    config.update(resolve_colab_runtime_overrides())
    config['output_dir'] = str(config['output_dir'])

    if task == 'lora_train_eval':
        config['per_device_train_batch_size'] = 1
        config['per_device_eval_batch_size'] = 1
        config['gradient_accumulation_steps'] = max(int(config.get('gradient_accumulation_steps', 8)), 8)

    colab_config_path = Path('configs') / f'colab_{task}.yaml'
    with open(colab_config_path, 'w', encoding='utf-8') as handle:
        yaml.safe_dump(config, handle, sort_keys=False)

    return colab_config_path


config_paths = {}
if RUN_BASELINE:
    config_paths['inference_only'] = build_colab_config('inference_only')
if RUN_LORA:
    config_paths['lora_train_eval'] = build_colab_config('lora_train_eval')

print('Prepared Colab configs:')
for task, config_path in config_paths.items():
    print(f'\n=== {task} :: {config_path} ===')
    print(Path(config_path).read_text(encoding='utf-8'))

print('Resolved runtime overrides:')
print(json.dumps(resolve_colab_runtime_overrides(), indent=2))

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import time


def run_pipeline(task: str, config_path: Path) -> None:
    script_path = 'scripts/run_inference_pipeline.sh' if task == 'inference_only' else 'scripts/run_lora_pipeline.sh'
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'

    print(f'\n===== START {task} =====')
    start = time.time()
    process = subprocess.Popen(
        ['bash', script_path, str(config_path)],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')

    return_code = process.wait()
    duration_s = time.time() - start
    print(f'===== END {task} ({duration_s / 60.0:.1f} min) =====')
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, process.args)


for task, config_path in config_paths.items():
    run_pipeline(task, Path(config_path))

In [ ]:
from pathlib import Path
import json
import subprocess
import pandas as pd
import yaml
from IPython.display import Image, display


def resolve_results_dir(task: str) -> Path:
    config_path = Path('configs') / f'colab_{task}.yaml'
    cfg = yaml.safe_load(config_path.read_text(encoding='utf-8'))
    root = Path(cfg['output_dir'])
    if task == 'inference_only':
        return root / 'overall_results'
    return root / 'checkpoint-final' / 'overall_results'


def generate_and_show_figures(task: str) -> dict:
    results_dir = resolve_results_dir(task)
    subprocess.run(
        ['python', '-m', 'scripts.generate_eval_report_figures', '--results-dir', str(results_dir)],
        check=True,
    )

    metrics = json.loads((results_dir / 'overall_metrics.json').read_text(encoding='utf-8'))
    print(f'\n=== {task} overall metrics ===')
    print(json.dumps(metrics, indent=2))

    fig_dir = results_dir / 'report_figures'
    for fig_path in sorted(fig_dir.glob('*.png')):
        print(f'\n[{task}] {fig_path.name}')
        display(Image(filename=str(fig_path)))

    return metrics


summary_rows = []
metric_names = ['exact_match_rate', 'avg_cer', 'avg_edit_score', 'avg_bleu4', 'avg_latency_s', 'num_samples']
for task in config_paths:
    metrics = generate_and_show_figures(task)
    row = {'task': task}
    for metric_name in metric_names:
        row[metric_name] = metrics.get(metric_name)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

if {'inference_only', 'lora_train_eval'}.issubset(config_paths):
    baseline = summary_df.loc[summary_df['task'] == 'inference_only'].iloc[0]
    lora = summary_df.loc[summary_df['task'] == 'lora_train_eval'].iloc[0]
    comparison_df = pd.DataFrame(
        [
            {'metric': 'exact_match_rate', 'baseline': baseline['exact_match_rate'], 'lora': lora['exact_match_rate'], 'delta': lora['exact_match_rate'] - baseline['exact_match_rate']},
            {'metric': 'avg_cer', 'baseline': baseline['avg_cer'], 'lora': lora['avg_cer'], 'delta': lora['avg_cer'] - baseline['avg_cer']},
            {'metric': 'avg_edit_score', 'baseline': baseline['avg_edit_score'], 'lora': lora['avg_edit_score'], 'delta': lora['avg_edit_score'] - baseline['avg_edit_score']},
            {'metric': 'avg_bleu4', 'baseline': baseline['avg_bleu4'], 'lora': lora['avg_bleu4'], 'delta': lora['avg_bleu4'] - baseline['avg_bleu4']},
            {'metric': 'avg_latency_s', 'baseline': baseline['avg_latency_s'], 'lora': lora['avg_latency_s'], 'delta': lora['avg_latency_s'] - baseline['avg_latency_s']},
        ]
    )
    print('\n=== Baseline vs LoRA comparison ===')
    display(comparison_df)
